In [29]:
import numpy as np
import pandas as pd
import pickle
import os
import io
import tarfile
import gzip
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression

In [ ]:
X_PATH = [f'../data/feat_matrix/X/X{i}.pkl' for i in range(1, 12)]
Y_PATH = [f'../data/feat_matrix/Y/Y{i}.pkl' for i in range(1, 12)]

In [4]:
X = []
for x_path in X_PATH:
    with open(x_path, 'rb') as f:
        l = pickle.load(f)
    X = X + l

Y = []
for y_path in Y_PATH:
    with open(y_path, 'rb') as f:
        l = pickle.load(f)
    Y = Y + l

In [16]:
len(X), type(X), type(X[0]), type(X[0][0]), X[0].shape

(2424, list, numpy.ndarray, numpy.ndarray, (4599, 90))

In [5]:
df = pd.DataFrame()
df_id = pd.DataFrame()
labels = pd.DataFrame()

for i, x in tqdm(enumerate(X), total=len(X)):
    sample_id = pd.DataFrame([i] * x.shape[0], columns=['image_id'])
    sample = pd.DataFrame(x, columns=[f'feat_{i}' for i in range(x.shape[1])])

    df_id = pd.concat([df_id, sample_id], axis=0, ignore_index=True)
    df = pd.concat([df, sample], axis=0, ignore_index=True)
df.insert(0, 'image_id', df_id['image_id'])

for y in tqdm(Y, total=len(Y)):
    label = pd.DataFrame(y, columns=['label'])
    labels = pd.concat([labels, label], axis=0, ignore_index=True)
df.insert(1, 'label', labels['label'])

100%|██████████| 2424/2424 [00:06<00:00, 397.09it/s]


In [6]:
df_path = '../data/feat_matrix/Manipulate-Image-Features.pkl'
if not os.path.exists(df_path):
    df.to_pickle(df_path)
else:
    with open(df_path, 'rb') as f:
        df = pickle.load(f)

In [2]:
df_path = Path('../data/feat_matrix/Manipulate-Image-Features.pkl')
archive_path = df_path.with_name(df_path.stem + '.tar.gz')

if not os.path.exists(archive_path):
    with tarfile.open(archive_path, 'w:gz') as tar:
        tar.add(df_path, arcname=df_path.name)
else:
    with tarfile.open(archive_path, 'r:gz') as tar:
        pkl_members = [m for m in tar.getmembers()
                       if m.name.endswith('.pkl') and m.isfile()]
        
        if not pkl_members:
            raise ValueError('No .pkl file found in archive')
        
        member = pkl_members[0]
        print(f'Extracting: {member.name!r} size={member.size} bytes')

        if member.size == 0:
            raise ValueError('Pkl member has 0 bytes -- archive is corrupt, delete and re-run')
        
        f = tar.extractfile(member)
        if f is None:
            raise ValueError('Could not extract .pkl file')
            
        data = f.read()

    if data[:2] == b'\x1f\x8b':
        data = gzip.decompress(data)

    df = pickle.loads(data)

    if not isinstance(df, pd.DataFrame):
        raise TypeError(f'Expected DataFrame, got {type(df)}')

Extracting: 'Manipulate-Image-Features.pkl' size=7694127320 bytes


In [4]:
df.head()

,image_id,label,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,...,feat_80,feat_81,feat_82,feat_83,feat_84,feat_85,feat_86,feat_87,feat_88,feat_89
0,0,0,0.606445,0.001953,0.0,0.0,0.0,0.0,0.0,0.0,...,0.064453,0.263672,1.727823,0.852311,0.168868,0.581885,0.691629,-0.247261,0.835812,0.853667
1,0,0,0.634766,0.001953,0.0,0.0,0.0,0.0,0.0,0.0,...,0.071289,0.261719,1.480847,0.845447,0.198830,0.624428,2.182204,-0.517785,0.817380,0.832598
2,0,0,0.635742,0.001953,0.0,0.0,0.0,0.0,0.0,0.0,...,0.061523,0.257812,1.843750,0.891489,0.178098,0.617363,3.087025,-0.062009,0.867658,0.894150
3,0,0,0.609375,0.001953,0.0,0.0,0.0,0.0,0.0,0.0,...,0.055664,0.244141,2.644153,0.841295,0.151001,0.560503,2.269908,-0.053270,0.814864,0.895385
4,0,0,0.616211,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.072266,0.256836,2.305444,0.842433,0.156008,0.552632,2.916367,-0.637114,0.826557,0.880141


In [5]:
df.describe()

,image_id,label,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,...,feat_80,feat_81,feat_82,feat_83,feat_84,feat_85,feat_86,feat_87,feat_88,feat_89
count,1.055436e+07,1.055436e+07,1.055436e+07,1.055436e+07,1.055436e+07,1.055436e+07,1.055436e+07,1.055436e+07,1.055436e+07,1.055436e+07,...,1.055436e+07,1.055436e+07,1.055436e+07,1.055436e+07,1.055436e+07,1.055436e+07,1.055436e+07,1.055436e+07,1.055436e+07,1.055436e+07
mean,1.250537e+03,7.219986e-02,6.521769e-01,2.485597e-02,7.339128e-03,2.945699e-03,1.330985e-03,6.354804e-04,3.197079e-04,1.648772e-04,...,1.005507e-01,2.237618e-01,1.983193e+01,8.773438e-01,2.411743e-01,5.982279e-01,2.399630e+00,7.290698e-02,8.169079e-01,8.169390e-01
std,6.606482e+02,2.588186e-01,1.872776e-01,4.080331e-02,1.719341e-02,8.959629e-03,5.219642e-03,3.152308e-03,2.008374e-03,1.303800e-03,...,1.279311e-01,9.591482e-02,5.644489e+01,1.346482e-01,2.219798e-01,2.407898e-01,1.890937e+01,1.528308e+00,1.786096e-01,2.661332e-01
min,0.000000e+00,0.000000e+00,1.562500e-02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,-6.672269e-01,2.398262e-02,1.426502e-02,-2.001952e+00,-3.190631e+01,-8.020482e-01,0.000000e+00
25%,6.390000e+02,0.000000e+00,5.371094e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,4.492187e-02,1.591797e-01,5.090726e-01,8.256626e-01,8.494701e-02,4.062317e-01,-6.104338e-01,-3.957306e-01,7.683660e-01,7.367085e-01
50%,1.302000e+03,0.000000e+00,6.445312e-01,9.765625e-04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,6.542969e-02,2.324219e-01,2.534274e+00,9.260694e-01,1.591178e-01,5.982696e-01,0.000000e+00,1.511787e-02,8.682999e-01,9.601067e-01
75%,1.845000e+03,0.000000e+00,7.890625e-01,3.515625e-02,3.906250e-03,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,9.179687e-02,2.978516e-01,1.383770e+01,9.712907e-01,3.202989e-01,8.058323e-01,1.095274e+00,5.217029e-01,9.256641e-01,9.935417e-01
max,2.423000e+03,1.000000e+00,1.000000e+00,4.677734e-01,3.105469e-01,1.943359e-01,1.826172e-01,1.425781e-01,1.152344e-01,8.984375e-02,...,1.000000e+00,6.767578e-01,2.700012e+03,1.000000e+00,1.000000e+00,1.000000e+00,1.017006e+03,3.190631e+01,9.999250e-01,9.997502e-01


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10554356 entries, 0 to 10554355
Data columns (total 92 columns):
 #   Column    Dtype  
---  ------    -----  
 0   image_id  int64  
 1   label     uint8  
 2   feat_0    float64
 3   feat_1    float64
 4   feat_2    float64
 5   feat_3    float64
 6   feat_4    float64
 7   feat_5    float64
 8   feat_6    float64
 9   feat_7    float64
 10  feat_8    float64
 11  feat_9    float64
 12  feat_10   float64
 13  feat_11   float64
 14  feat_12   float64
 15  feat_13   float64
 16  feat_14   float64
 17  feat_15   float64
 18  feat_16   float64
 19  feat_17   float64
 20  feat_18   float64
 21  feat_19   float64
 22  feat_20   float64
 23  feat_21   float64
 24  feat_22   float64
 25  feat_23   float64
 26  feat_24   float64
 27  feat_25   float64
 28  feat_26   float64
 29  feat_27   float64
 30  feat_28   float64
 31  feat_29   float64
 32  feat_30   float64
 33  feat_31   float64
 34  feat_32   float64
 35  feat_33   float64
 36  feat_3

In [10]:
nonManip_n = int(len(df[df['label'] == 1]) * 1.5)
df_non = df[df['label'] == 0].sample(n=nonManip_n, random_state=21)
len(df_non)

1143034

In [12]:
df_non.head()

,image_id,label,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,...,feat_80,feat_81,feat_82,feat_83,feat_84,feat_85,feat_86,feat_87,feat_88,feat_89
7129573,1661,0,0.819336,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,...,0.056641,0.157227,0.311492,0.979028,0.266206,0.851512,-0.611666,0.468168,0.974744,0.884028
192408,77,0,0.935547,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,...,0.346680,0.045898,0.101815,0.907578,0.565427,0.951512,0.280342,-0.231713,0.889300,0.353585
5122224,1266,0,0.572266,0.009766,0.000000,0.000000,0.0,0.0,0.0,0.0,...,0.048828,0.277344,3.319556,0.909782,0.131407,0.527220,6.943521,-1.756583,0.903272,0.947475
5337967,1313,0,0.599609,0.005859,0.000000,0.000000,0.0,0.0,0.0,0.0,...,0.092773,0.368164,4.052419,0.861244,0.142321,0.482123,3.103778,1.422689,0.847664,0.936045
8717337,1968,0,0.583984,0.034180,0.011719,0.001953,0.0,0.0,0.0,0.0,...,0.050781,0.218750,10.914315,0.975628,0.094707,0.494592,0.171443,-0.334801,0.939011,0.995551


In [17]:
df_bal = pd.concat([df_non, df[df['label'] == 1]], axis=0, ignore_index=True)
df_bal = df_bal.sample(frac=1, random_state=21).reset_index(drop=True)
df_bal.head(10)

,image_id,label,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,...,feat_80,feat_81,feat_82,feat_83,feat_84,feat_85,feat_86,feat_87,feat_88,feat_89
0,267,0,0.707031,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.025391,0.065430,5.775202,0.980930,0.120926,0.440023,-1.075825,0.559220,0.971305,0.993365
1,629,0,0.846680,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.022461,0.120117,0.320565,0.964445,0.275095,0.859073,-0.410978,0.050078,0.930902,0.820914
2,2132,1,0.719727,0.041992,0.015625,0.012695,0.000977,0.002930,0.000977,0.000977,...,0.279297,0.105469,82.446573,0.971680,0.621504,0.701711,0.247272,-1.437000,0.946297,0.999308
3,1055,1,0.916992,0.007812,0.008789,0.003906,0.004883,0.006836,0.000977,0.000000,...,0.535156,0.045898,43.724798,0.970761,0.910335,0.932347,10.360305,-3.481053,0.894176,0.998718
4,88,1,0.402344,0.057617,0.019531,0.025391,0.022461,0.014648,0.011719,0.003906,...,0.052734,0.237305,215.515121,0.795067,0.046212,0.214224,1.146800,1.127843,0.757164,0.998118
5,1257,1,0.564453,0.020508,0.005859,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.032227,0.184570,3.741935,0.974799,0.106783,0.552755,1.372865,1.134784,0.924426,0.986701
6,2112,0,0.551758,0.051758,0.012695,0.000977,0.000000,0.000000,0.000000,0.000000,...,0.031250,0.126953,60.760081,0.968372,0.083167,0.369124,-1.426917,-0.449905,0.931151,0.998961
7,484,0,0.922852,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.187500,0.138672,0.175403,0.973433,0.473373,0.912298,0.246311,1.214989,0.916106,0.769018
8,185,1,0.514648,0.032227,0.027344,0.010742,0.004883,0.003906,0.000000,0.000000,...,0.033203,0.149414,90.104839,0.936031,0.040599,0.191083,-1.419181,0.198913,0.877216,0.998612
9,331,0,0.519531,0.046875,0.003906,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.041016,0.177734,17.864919,0.979444,0.061438,0.380290,-0.566068,0.496204,0.946178,0.997697


In [19]:
def minmax_norm(df, columns, eps=1e-6):
    df_norm = df.copy()
    for col in columns:
        if col in df_norm.columns:
            col_min, col_max = df_norm[col].min(), df_norm[col].max()
            norm = (df_norm[col] - col_min) / (col_max - col_min)
            df_norm[col] = norm * (1 - 2*eps) + eps

    return df_norm

In [21]:
norm_columns = df_bal.columns[2:]
df_norm = minmax_norm(df_bal, norm_columns)

In [22]:
df_norm.describe()

,image_id,label,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,...,feat_80,feat_81,feat_82,feat_83,feat_84,feat_85,feat_86,feat_87,feat_88,feat_89
count,1.905057e+06,1.905057e+06,1.905057e+06,1.905057e+06,1.905057e+06,1.905057e+06,1.905057e+06,1.905057e+06,1.905057e+06,1.905057e+06,...,1.905057e+06,1.905057e+06,1.905057e+06,1.905057e+06,1.905057e+06,1.905057e+06,1.905057e+06,1.905057e+06,1.905057e+06,1.905057e+06
mean,1.228213e+03,4.000001e-01,6.246003e-01,6.145964e-02,3.726962e-02,1.893404e-02,9.872438e-03,5.875479e-03,4.686974e-03,2.877190e-03,...,9.224095e-02,3.340372e-01,1.008023e-02,9.335319e-01,1.991083e-01,5.625983e-01,4.517770e-03,5.012294e-01,8.963944e-01,8.559270e-01
std,6.591855e+02,4.898981e-01,1.895931e-01,8.917780e-02,7.608887e-02,5.020945e-02,3.355145e-02,2.510858e-02,2.510471e-02,1.926758e-02,...,1.206244e-01,1.416444e-01,2.588684e-02,7.633705e-02,2.152047e-01,2.412295e-01,1.867672e-02,2.510975e-02,1.018554e-01,2.434085e-01
min,0.000000e+00,0.000000e+00,1.000000e-06,1.000000e-06,1.000000e-06,1.000000e-06,1.000000e-06,1.000000e-06,1.000000e-06,1.000000e-06,...,1.000000e-06,1.000000e-06,1.000000e-06,1.000000e-06,1.000000e-06,1.000000e-06,1.000000e-06,1.000000e-06,1.000000e-06,1.000000e-06
25%,6.360000e+02,0.000000e+00,5.044955e-01,1.000000e-06,1.000000e-06,1.000000e-06,1.000000e-06,1.000000e-06,1.000000e-06,1.000000e-06,...,4.101654e-02,2.375945e-01,2.908040e-04,9.102610e-01,5.344917e-02,3.683241e-01,1.302292e-03,4.929979e-01,8.723845e-01,8.407133e-01
50%,1.256000e+03,0.000000e+00,6.173824e-01,1.043939e-02,1.000000e-06,1.000000e-06,1.000000e-06,1.000000e-06,1.000000e-06,1.000000e-06,...,6.054775e-02,3.428575e-01,1.608741e-03,9.612392e-01,1.186487e-01,5.522805e-01,1.965605e-03,5.003414e-01,9.252396e-01,9.801332e-01
75%,1.792000e+03,1.000000e+00,7.572422e-01,9.812189e-02,3.673562e-02,5.026116e-03,1.000000e-06,1.000000e-06,1.000000e-06,1.000000e-06,...,8.593833e-02,4.406016e-01,8.691962e-03,9.839521e-01,2.622303e-01,7.627891e-01,3.178963e-03,5.091977e-01,9.558562e-01,9.962547e-01
max,2.423000e+03,1.000000e+00,9.999990e-01,9.999990e-01,9.999990e-01,9.999990e-01,9.999990e-01,9.999990e-01,9.999990e-01,9.999990e-01,...,9.999990e-01,9.999990e-01,9.999990e-01,9.999990e-01,9.999990e-01,9.999990e-01,9.999990e-01,9.999990e-01,9.999990e-01,9.999990e-01


In [55]:
kf = KFold(n_splits=5)
X_df = df_norm.iloc[:, 2:]
y = df_norm['label']

In [70]:
for train_index, test_index in kf.split(X_df):
    X_train, X_test = X_df.iloc[train_index, :], X_df.iloc[test_index, :]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = LogisticRegression(max_iter=1000)

    model.fit(X_train, y_train)

    score = model.score(X_test, y_test)
    print("Fold score:", score)

Fold score: 0.6805113749698172
Fold score: 0.6785166871384628
Fold score: 0.6790486363910754
Fold score: 0.6800039893861332
Fold score: 0.6790696331601975
